In [ ]:
from dynesty import NestedSampler
import numpy as np
from scipy.stats import beta, expon
import os

mining_data = np.loadtxt(os.path.join('data','coal_mining_accident_data.dat'))
flattened_data = np.ndarray.flatten(mining_data, order="C")
flattened_data_cumulative = np.cumsum(flattened_data)
print(np.sum(flattened_data_cumulative < 20000))

total_period = 40550
total_events = 191

number_of_accidents = np.arange(0, len(flattened_data_cumulative), 1)

beta_param = 200

def log_likelihood(heights, change_points):
    """
    Returns the value of the log-likelihood at a set of heights given a set of change_points.

    Arguments:
        heights (np.array) - An array of heights at which to evaluate the log-likelihood.
        change_points (np.array) - An array of change points with length between zero and 30.
    
    Returns:
        ll (float) - The value of the log-likelihood.
    """
    ns = []
    for i in range(len(heights)):
        if i == 0:
            ns.append(np.sum(flattened_data_cumulative < s[0]))
        else:
            ns.append(np.sum(s[i - 1] < flattened_data_cumulative and flattened_data_cumulative < s[i]))
    ns = np.array()
    Ts = np.array()
    log_heights = np.log(heights)

    ll = ns * log_heights - heights * Ts

    return ll

def prior_transform(u):
    """
    Transforms the uniform random variables `u ~ Unif[0., 1.)`
    to the parameters of interest.
     - h_0 and h_1 are drawn from a gamma distribution with rate beta_param = 200 days and alpha_param = 1
     - s_1 is drawn from a Dirichlet distribution with k=2 and alpha parameters equal to 2. 
    """
    x = np.array(u)
    x[0] = beta.ppf(u[0],2,2)
    x[1] = expon.ppf(u[1], scale=1/beta_param)
    x[2] = expon.ppf(u[2], scale=1/beta_param)
    return x

sampler = NestedSampler(log_likelihood, prior_transform, ndim, nlive=500)